# IoT Sensor ETL Pipeline — Minimal Adaptation

This notebook transforms openSenseMap box/sensor payloads into a measurement-grain dataset. The source `/boxes` response contains box metadata (`_id`, `createdAt`, `updatedAt`, `name`, `exposure`, `model`, `description`, `image`, `lastMeasurementAt`, `grouptag`, `loc`, `currentLocation`, and `weblink`) plus a nested `sensors` array. Each sensor can contain `_id`, `title`, `unit`, `sensorType`, `icon`, and `lastMeasurement` (`createdAt`, `value`).

The ETL row grain is **one row per historical sensor measurement**. Box and sensor attributes are repeated on each measurement row so the dataset remains analytics-friendly and can be filtered by device, sensor, location, model, and unit. `measurement_time` and `value` from the measurement endpoint are kept distinct from the sensor's latest reading metadata.

**Storage:** local Parquet is the primary sink; Kaggle publishing remains optional. **Processing:** Spark is used for typed transformations, deduplication, ordering, and Parquet output when available; pandas/PyArrow remains a fallback. API requests remain network-bound and are not parallelized by Spark.

The `/boxes` preview may return thousands of boxes; the ETL only processes up to `API_CONFIG.max_boxes` matching boxes per run. Keep the default small on a memory-constrained machine and increase it only after testing request volume and runtime.

**Source:** [openSenseMap API](https://api.opensensemap.org/). Discovery responses may contain many boxes, so extraction processes at most `API_CONFIG.max_boxes` matching boxes per run. Measurement requests use UTC `Z` timestamps and windows of no more than 30 days, with per-request and overall timeouts.

## Pipeline Reliability & Idempotency

The pipeline is designed for repeatable execution and data integrity. Each run retrieves a configurable overlap window to capture delayed or late-arriving sensor measurements, merges the new observations with the existing Parquet dataset, and deduplicates records using stable observation identifiers or composite keys.

Processed data is written to a temporary file, validated, and atomically replaces the existing output only after the pipeline completes successfully. If API extraction times out, measurement requests fail, or validation detects an error, the existing dataset remains unchanged, preventing incomplete runs from corrupting previously persisted data.

## API Extraction & Box Discovery

The initial API connectivity check retrieves a bounded response from the openSenseMap `/boxes` endpoint, converts the response into a Pandas DataFrame, and displays a sample of up to five records for inspection.

Box discovery uses the configured measurement `date` and `phenomenon` parameters together, in accordance with the openSenseMap API requirements. Users can optionally specify known public box IDs through `STATION_CONFIG.box_ids` to restrict extraction to selected stations. When no box IDs are configured, the pipeline discovers eligible boxes based on the requested phenomenon and extraction parameters.

The `reversed_df` DataFrame is populated with actual API-shaped measurement records during pipeline execution. The schema-validation sample is used exclusively for testing transformations and is not substituted for live extraction data.

## Distributed Processing & Runtime Configuration

The pipeline uses PySpark 3.5.x for local data processing and Parquet persistence, with Pandas available as a fallback for compatible transformations and smaller workloads.

The recommended runtime environment is:

| Component | Configuration |
|---|---|
| Python | 3.10–3.14 |
| PySpark | 3.5.x |
| Java | 17 |
| Spark master | `local[2]` |
| Spark driver memory | 1 GB |
| Shuffle partitions | Conservatively configured |

Spark handles local data transformations and Parquet processing. API extraction remains network-bound and is performed through HTTP requests; Spark is not used to parallelize API calls.

The runtime configuration is intentionally conservative to accommodate resource-constrained environments while maintaining compatibility with the Spark processing workflow.


## Environment setup (run in your WSL terminal, not in this notebook)

PySpark 3.5.x should be run with a compatible Python version. Do **not** install it into the current Python 3.14 kernel. From the project directory, create a dedicated Python 3.12 environment, then select its `iot-spark` kernel in Jupyter:

```bash
# Check whether Python 3.12 is installed first
python3.12 - 3.14 --version

# From ~/data-engineering/iot_sensor_etl
python3.14 -m venv .venv-iot-py314
source .venv-iot-py314/bin/activate
python -m pip install --upgrade pip
python -m pip install "pyspark==3.5.6" pandas numpy requests pyarrow matplotlib ipykernel
python -m ipykernel install --user --name iot-spark --display-name "Python (IoT Spark 3.5)"
```

If `python3.12` is not installed, install it in WSL first or use an existing compatible Python 3.10–3.12 environment. Java 17 is appropriate. After selecting the kernel, restart it and run the notebook from the top. The ETL has a pandas fallback if Spark cannot start.


In [1]:
import json
import os
import random
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass, field
from datetime import date, datetime, timedelta
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple

# Third-party libraries
import numpy as np
import pandas as pd
import requests

# Apache Spark is optional at import time: use pandas/PyArrow if PySpark is unavailable.
try:
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    from pyspark.sql.types import (
        StructType, StructField, StringType, TimestampType, DoubleType, IntegerType
    )
except ImportError:
    SparkSession = None
    F = None
    StructType = StructField = StringType = TimestampType = DoubleType = IntegerType = None

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


In [2]:
@dataclass
class APIConfig:
    """API configuration for public IoT sensor readings from openSenseMap."""
    use_mock: bool = False
    base_url: str = "https://api.opensensemap.org"
    phenomenon: str = "temperature"
    # Maximum number of matching boxes processed per run (not total boxes returned by API).
    max_boxes: int = 5
    timeout_seconds: int = 600
    overall_timeout_seconds: int = 120
    request_window_days: int = 30

    def __post_init__(self):
        if self.max_boxes <= 0:
            raise ValueError("max_boxes must be positive")
        if self.timeout_seconds <= 0 or self.overall_timeout_seconds <= 0:
            raise ValueError("Timeout values must be positive")
        if not 1 <= self.request_window_days <= 30:
            raise ValueError("request_window_days must be between 1 and 30")


In [3]:
@dataclass
class StationConfig:
    """IoT sensor selection and incremental ingestion configuration."""
    # Optional explicit box IDs. Empty means discover boxes through openSenseMap.
    box_ids: List[str] = field(default_factory=list)
    phenomenon: str = "temperature"
    
    # First run starts with a recent, bounded window. Set this explicitly for a deliberate backfill.
    earliest_date: str = field(default_factory=lambda: (pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=7)).strftime("%Y-%m-%d"))
    
    # Re-fetch a small overlap to catch delayed measurements; deduplication makes reruns safe.
    overlap_days: int = 1
    
    # Official API discovery parameters.
    name: Optional[str] = None
    limit: int = 5
    format: str = "json"
    minimal: bool = False
    full: bool = False
    classify: bool = False

    # Optional filters supported by the API.
    grouptag: Optional[str] = None
    model: Optional[str] = None
    exposure: Optional[str] = None
    near: Optional[str] = None
    maxDistance: Optional[int] = None
    bbox: Optional[str] = None

    # Optional explicit discovery timestamps.
    date: Optional[str] = None


In [4]:
@dataclass
class BadConfig:
    def __init__(self, station_codes=[]):
        self.station_codes = station_codes

# class BadConfig:
#     station_codes : list[str] = []

a = BadConfig()
b = BadConfig()

a.station_codes.append("ALERT")
print(a.station_codes)
print(b.station_codes)


# Good Configuration : Fresh List per instance
@dataclass
class GoodConfig:
    station_codes : list[str] = field(default_factory = list)

x = GoodConfig()
y = GoodConfig()

x.station_codes.append("ALERT")
print(x.station_codes)
print(y.station_codes)




['ALERT']
['ALERT']
['ALERT']
[]


In [5]:
@dataclass
class KaggleConfig:
    """Output configuration retained from the original pipeline; local Parquet is the primary sink."""
    dataset_slug: str = "public-iot-sensor-readings"
    input_csv: str = ""
    output_dir: Path = field(default_factory=lambda: Path("data/processed"))
    mock_output_dir: Path = field(default_factory=lambda: Path("data/mock"))
    output_filename: str = "iot_sensor_readings.parquet"
    mock_output_filename: str = "iot_sensor_readings_mock.parquet"
    metadata_filename: str = "dataset-metadata.json"
    
    #Metadata
    title: str = "Public IoT Sensor Readings"
    keywords: list = field(default_factory=lambda: ["iot", "sensors", "time series", "environment", "open data"])
    geospatial_coverage: str = "Locations provided by public openSenseMap sensors"
    update_frequency: str = "Daily"
    license_name: str = "CM - DE - 2026"

    output_csv_path: Path = field(init=False)
    metadata_path: Path = field(init=False)
    output_parquet_path: Path = field(init=False)
    mock_output_path: Path = field(init=False)

    def __post_init__(self):
        self.output_csv_path = self.output_dir / self.output_filename
        self.metadata_path = self.output_dir / self.metadata_filename
        self.output_parquet_path = self.output_dir / self.output_filename
        self.mock_output_path = self.mock_output_dir / self.mock_output_filename

API_CONFIG = APIConfig()
STATION_CONFIG = StationConfig()
KAGGLE_CONFIG = KaggleConfig()


In [6]:
print(API_CONFIG)
print(STATION_CONFIG)
print(f"Parquet output: {KAGGLE_CONFIG.output_parquet_path}")

APIConfig(use_mock=False, base_url='https://api.opensensemap.org', phenomenon='temperature', max_boxes=5, timeout_seconds=600, overall_timeout_seconds=120, request_window_days=30)
StationConfig(box_ids=[], phenomenon='temperature', earliest_date='2026-11-11', overlap_days=1, name=None, limit=5, format='json', minimal=False, full=False, classify=False, grouptag=None, model=None, exposure=None, near=None, maxDistance=None, bbox=None, date=None)
Parquet output: data/processed/iot_sensor_readings.parquet


In [7]:
# Start Spark conservatively to preserve memory. Increase workers only after benchmarking.
# API extraction remains network-bound; Spark parallelizes local transformations, not the HTTP requests.
SPARK_WORKER_CORES = 3
SPARK_SHUFFLE_PARTITIONS = max(2, SPARK_WORKER_CORES * 2)

spark = None
try:
    spark = (
        SparkSession.builder
        .appName("IoT Sensor ETL Pipeline")
        .master(f"local[{SPARK_WORKER_CORES}]")
        .config("spark.driver.memory", "1g")
        .config("spark.default.parallelism", str(SPARK_WORKER_CORES))
        .config("spark.sql.shuffle.partitions", str(SPARK_SHUFFLE_PARTITIONS))
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.sql.adaptive.enabled", "true")
        .getOrCreate()
    )
    # Tiny action catches runtime/serialization incompatibilities early.
    spark.range(1).count()
    print(
        f"Spark ready: {spark.version} | master={spark.sparkContext.master} | "
        f"shuffle partitions={SPARK_SHUFFLE_PARTITIONS}"
    )
except Exception as exc:
    if spark is not None:
        try:
            spark.stop()
        except Exception:
            pass
    spark = None
    print(f"Spark unavailable in this Python/JDK runtime; pandas fallback enabled. Details: {type(exc).__name__}: {exc}")


26/11/18 10:19:15 WARN Utils: Your hostname, DESKTOP-VFJRAO4 resolves to a loopback address: 127.0.1.1; using 172.20.76.5 instead (on interface eth0)
26/11/18 10:19:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/18 10:19:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
[Stage 0:>                                                          (0 + 3) / 3]

Spark ready: 3.5.1 | master=local[3] | shuffle partitions=6


In [8]:
# API payload check: preview rows are box records with a nested sensors array, not one row per measurement.
# A preview failure is reported but does not prevent the remaining ETL functions from being defined.
preview_boxes_df = pd.DataFrame()
try:
    preview_end = pd.Timestamp.now(tz="UTC")
    preview_start = preview_end - pd.Timedelta(days=7)
    phenomenon_query = (
        "Temperatur"
        if STATION_CONFIG.phenomenon.strip().lower() in {"temperature", "temperatur", "temp"}
        else STATION_CONFIG.phenomenon
    )
    preview_params = {
        "limit": 5,
        "date": (
            f"{preview_start.strftime('%Y-%m-%dT%H:%M:%SZ')},"
            f"{preview_end.strftime('%Y-%m-%dT%H:%M:%SZ')}"
        ),
        "phenomenon": phenomenon_query,
        "format": "json",
        "minimal": "false",
    }
    preview_response = requests.get(
        f"{API_CONFIG.base_url.rstrip('/')}/boxes",
        params=preview_params,
        timeout=(5, API_CONFIG.timeout_seconds),
    )
    preview_response.raise_for_status()
    preview_payload = preview_response.json()
    if isinstance(preview_payload, dict):
        preview_payload = preview_payload.get("data", preview_payload.get("boxes", []))
    if not isinstance(preview_payload, list):
        raise ValueError("Unexpected openSenseMap /boxes response; expected a JSON list of boxes.")
    preview_boxes_df = pd.json_normalize(preview_payload, sep="_")
    print(f"API preview: {len(preview_boxes_df)} box record(s); DataFrame shape={preview_boxes_df.shape}")
    print("Nested sensor keys:", sorted({key for box in preview_payload[:5] if isinstance(box, dict) for sensor in (box.get("sensors") or []) if isinstance(sensor, dict) for key in sensor.keys()}))
    display(preview_boxes_df.head(5))
except (requests.RequestException, ValueError) as exc:
    print(f"API preview unavailable ({type(exc).__name__}: {exc}). ETL definitions are still available; retry the preview later.")


API preview: 2636 box record(s); DataFrame shape=(2636, 16)
Nested sensor keys: ['_id', 'icon', 'lastMeasurement', 'sensorType', 'title', 'unit']


,_id,createdAt,updatedAt,name,exposure,model,sensors,description,image,lastMeasurementAt,grouptag,loc,currentLocation_coordinates,currentLocation_type,currentLocation_timestamp,weblink
0,578207d56fea661300861f3b,2022-03-30T11:25:43.223Z,2026-11-18T16:22:46.368Z,emp-wetter,outdoor,homeEthernet,"[{'_id': '578207d56fea661300861f3d', 'title': 'UV-Intensität', 'unit': 'μW/cm²', 'sensorType': 'VEML6070', 'lastMeasurement': {'createdAt': '2026-11-18T16:22:46.356Z', 'value': '0'}}, {'_id': '578207d56fea661300861f3e', 'title': 'Beleuchtungsstärke', 'unit': 'lx', 'sensorType': 'TSL45315', 'lastMeasurement': {'createdAt': '2026-11-18T16:22:46.356Z', 'value': '0'}}, {'_id': '578207d56fea661300861f3f', 'title': 'Luftdruck', 'unit': 'hPa', 'sensorType': 'BMP280', 'lastMeasurement': {'createdAt': '2026-11-18T16:22:46.356Z', 'value': '975'}}, {'_id': '578207d56fea661300861f40', 'title': 'rel. Luftfeuchte', 'unit': '%', 'sensorType': 'HDC1008', 'lastMeasurement': {'createdAt': '2026-11-18T16:22:46.356Z', 'value': '96.18'}}, {'_id': '578207d56fea661300861f41', 'title': 'Temperatur', 'unit': '°C', 'sensorType': 'HDC1008', 'lastMeasurement': {'createdAt': '2026-11-18T16:22:46.356Z', 'value': '8.14'}}, {'_id': '57a990b9d46eb21200e231c4', 'sensorType': 'FWG14-V', 'unit': 'm/s', 'title': 'Windgeschwindigkeit', 'lastMeasurement': {'createdAt': '2026-11-18T16:22:46.356Z', 'value': '0'}}, {'_id': '57ab3d3791673313004b003d', 'sensorType': 'FWG14-D', 'unit': 'lx', 'title': 'Dämmerung', 'lastMeasurement': {'createdAt': '2026-11-18T16:22:46.356Z', 'value': '0'}, 'icon': 'osem-windspeed'}, {'_id': '57ab3d3791673313004b003e', 'sensorType': 'FWG14-R', 'unit': 'mm /m² Tag', 'title': 'Regen', 'lastMeasurement': {'createdAt': '2026-11-18T16:22:46.356Z', 'value': '7.711'}, 'icon': 'osem-sprinkles'}, {'_id': '57ab3d3791673313004b003f', 'sensorType': 'FWG14-SO', 'unit': 'kLx', 'title': 'Licht Sonne Ost', 'lastMeasurement': {'createdAt': '2026-11-18T16:22:46.356Z', 'value': '0'}, 'icon': 'osem-brightness'}, {'_id': '57ab3d3791673313004b0040', 'sensorType': 'FWG14-SS', 'unit': 'kLx', 'title': 'Licht Sonne Süd', 'lastMeasurement': {'createdAt': '2026-11-18T16:22:46.356Z', 'value': '0'}, 'icon': 'osem-brightness'}, {'_id': '57ab3d3791673313004b0041', 'sensorType': 'FWG14-SW', 'unit': 'kLx', 'title': 'Licht Sonne West', 'lastMeasurement': {'createdAt': '2026-11-18T16:22:46.356Z', 'value': '0'}, 'icon': 'osem-brightness'}, {'_id': '57b601ddb17d681100196369', 'sensorType': 'FWG14-RI', 'unit': '°', 'title': 'Windrichtung', 'lastMeasurement': {'createdAt': '2026-11-18T16:22:46.356Z', 'value': '247.5'}}, {'_id': '57dc18b72fc31e1600a7ba37', 'sensorType': 'SM10E101', 'unit': 'Wh', 'title': 'TagesLeistung', 'lastMeasurement': {'createdAt': '2026-11-18T16:22:46.356Z', 'value': '0'}}, {'_id': '57e426728859c00011179c6f', 'sensorType': 'SM10E102', 'unit': 'W', 'title': 'LeistungAktuell', 'lastMeasurement': {'createdAt': '2026-11-18T16:22:46.356Z', 'value': '3.81178'}, 'icon': 'osem-brightness'}, {'_id': '57e7ff818859c0001124419d', 'title': 'Grundwasserstand', 'unit': 'cm', 'sensorType': 'Pegelsonde', 'icon': 'osem-windspeed', 'lastMeasurement': {'createdAt': '2026-11-18T16:22:46.356Z', 'value': '141.483'}}, {'_id': '590034635ba9e50011e6f265', 'icon': 'osem-particulate-matter', 'sensorType': 'UP30A001', 'unit': 'µg/m³', 'title': 'Feinstaub PM2,5', 'lastMeasurement': {'createdAt': '2026-11-18T16:22:46.356Z', 'value': '0.5'}}, {'_id': '590034635ba9e50011e6f266', 'icon': 'osem-particulate-matter', 'sensorType': 'UP30A002', 'unit': 'µg/m³', 'title': 'Feinstaub PM10', 'lastMeasurement': {'value': '2.9', 'createdAt': '2026-11-18T07:18:51.686Z'}}]",Aufbereitung der Messwerte erfolgt durch einen Arduino Uno.\nÜber eine Modbus TCP-Schnittstelle werden die Messwert an eine Mini SPS übertragen. Diese SPS überträgt dann diese und weitere Daten als JSON-Array im 60s Takt.,578207d56fea661300861f3b_ovyk87.jpg,2026-11-18T16:22:46.356Z,[],"[{'geometry': {'coordinates': [9.391277, 51.660264], 'type':

In [9]:
def load_csv(path : str) -> pd.DataFrame:
    """Load CSV file or return empty DataFrame if file does not exist
    Args:
        path (str) : Full path to CSV file
    Returns:
        pd.DataFrame : Loaded data, or empty DataFrame if file not found.

    Raises:
        pd.errors.ParserError : If the csv is malformed.
    """
    if os.path.exists(path):
        return pd.read_csv(path, low_memory=False, parse_dates=True, delimiter=",")
    return pd.DataFrame()

In [10]:
df_missing = load_csv("/temp/does_not_exist.csv")
print(df_missing)

if df_missing.empty:
    print("No existing Data, will run a full fetch from earliest date")

Empty DataFrame
Columns: []
Index: []
No existing Data, will run a full fetch from earliest date


In [11]:
# def determine_update_range(existing: pd.DataFrame) -> Tuple[bool, Optional[str]]:
#     """Choose a resumable start date, with overlap to catch late-arriving readings."""
#     if existing.empty:
#         print("No existing data found. Starting from the configured earliest date.")
#         return True, STATION_CONFIG.earliest_date

#     record_colname = "record_date" if "record_date" in existing.columns else "measurement_time" if "measurement_time" in existing.columns else None
#     if record_colname is None:
#         raise KeyError("Missing date column: expected 'record_date' or 'measurement_time'.")

#     dates = pd.to_datetime(existing[record_colname], errors="coerce", utc=True).dropna()
#     if dates.empty:
#         return True, STATION_CONFIG.earliest_date

#     # Re-read the most recent overlap period instead of assuming all readings arrived on time.
#     resume_from = dates.max() - pd.Timedelta(days=STATION_CONFIG.overlap_days)
#     earliest = pd.Timestamp(STATION_CONFIG.earliest_date, tz="UTC")
#     resume_from = max(resume_from, earliest)
#     start_date = resume_from.strftime("%Y-%m-%d")
#     print(f"Incremental fetch starts at {start_date} (includes overlap for idempotent upserts).")
#     return True, start_date

def determine_update_range(existing: pd.DataFrame, sensors_df: pd.DataFrame) -> Tuple[bool, Optional[str]]:
    """Choose a resumable start date, with overlap to catch late-arriving readings."""
    if existing.empty and sensors_df.empty:
        print("No existing box or sensor data found. Starting from the configured earliest date.")
        return True, STATION_CONFIG.earliest_date

    date_columns = []

    if "last_measurement_at" in existing.columns:
        date_columns.append(
            pd.to_datetime(existing["last_measurement_at"], errors="coerce", utc=True)
        )

    if "last_measurement_createdAt" in sensors_df.columns:
        date_columns.append(
            pd.to_datetime(sensors_df["last_measurement_createdAt"], errors="coerce", utc=True)
        )

    if not date_columns:
        raise KeyError(
            "Missing date columns: expected 'last_measurement_at' in boxes_df "
            "or 'last_measurement_createdAt' in sensors_df."
        )

    dates = pd.concat(date_columns, ignore_index=True).dropna()

    if dates.empty:
        return True, STATION_CONFIG.earliest_date

    # Re-read the most recent overlap period instead of assuming all readings arrived on time.
    resume_from = dates.max() - pd.Timedelta(days=STATION_CONFIG.overlap_days)
    earliest = pd.Timestamp(STATION_CONFIG.earliest_date, tz="UTC")
    resume_from = max(resume_from, earliest)

    start_date = resume_from.strftime("%Y-%m-%d")

    print(f"Incremental fetch starts at {start_date} (includes overlap for idempotent upserts).")
    return True, start_date

In [12]:
# Test 1 : No existing Data
should_update, start_date = determine_update_range(pd.DataFrame(), pd.DataFrame())

# Test 2 Old Existing data (covers only Jan 10 - 15)
# True , 2026-01-16 the day after the last known date

# Test 3 Recent data that already covers yesterday
# False , None

No existing box or sensor data found. Starting from the configured earliest date.


In [13]:
# API discovery is performed once by fetch_all_data() during the ETL run.
# Avoid a separate smoke-test call here because the boxes endpoint may return thousands of records.
print(f"Configured extraction cap: {API_CONFIG.max_boxes} matching boxes; request window: {API_CONFIG.request_window_days} days.")


Configured extraction cap: 5 matching boxes; request window: 30 days.


In [14]:
def generate_mock_api_data(station_code: str, start_date: str, num_days: int = 10) -> List[Dict]:
    """Generate clearly labelled sample IoT readings for offline pipeline tests only."""
    start = pd.to_datetime(start_date).date()
    records = []
    for i in range(num_days):
        obs_time = pd.Timestamp(start + timedelta(days=i), tz="UTC")
        records.append({
            "device_id": f"mock_{station_code}", "device_name": f"Mock sensor box {station_code}",
            "sensor_id": f"mock_sensor_{station_code}", "sensor_name": STATION_CONFIG.phenomenon,
            "measurement_type": STATION_CONFIG.phenomenon, "unit": "°C", "measurement_time": obs_time.isoformat(),
            "value": round(18 + random.uniform(-4, 8), 2), "latitude": 0.0, "longitude": 37.0,
        })
    return records


In [15]:
def preview_open_sensemap_boxes():
    """Fetch matching openSenseMap boxes once and display a preview DataFrame."""
    preview_end = pd.Timestamp.now(tz="UTC")
    preview_start = preview_end - pd.Timedelta(days=7)

    phenomenon = (
        "Temperatur"
        if STATION_CONFIG.phenomenon.strip().lower()
        in {"temperature", "temperatur", "temp"}
        else STATION_CONFIG.phenomenon
    )

    params = {
        "limit": STATION_CONFIG.limit,
        "date": STATION_CONFIG.date or (
            f"{preview_start.strftime('%Y-%m-%dT%H:%M:%SZ')},"
            f"{preview_end.strftime('%Y-%m-%dT%H:%M:%SZ')}"
        ),
        "phenomenon": phenomenon,
        "format": STATION_CONFIG.format,
        "minimal": str(STATION_CONFIG.minimal).lower(),
        "full": str(STATION_CONFIG.full).lower(),
        "classify": str(STATION_CONFIG.classify).lower(),
    }

    response = requests.get(
        f"{API_CONFIG.base_url.rstrip('/')}/boxes",
        params=params,
        timeout=(5, API_CONFIG.timeout_seconds),
    )
    response.raise_for_status()

    payload = response.json()
    if isinstance(payload, dict):
        payload = payload.get("data", payload.get("boxes", []))

    if not isinstance(payload, list):
        raise ValueError(
            "Unexpected openSenseMap /boxes response; expected a JSON list of boxes."
        )

    preview_boxes_df = pd.json_normalize(payload, sep="_")

    print(
        f"API preview: {len(preview_boxes_df)} box record(s); "
        f"DataFrame shape={preview_boxes_df.shape}"
    )
    display(preview_boxes_df.head(5))

    return preview_boxes_df


# Execute the function
preview_boxes_df = preview_open_sensemap_boxes()

API preview: 2636 box record(s); DataFrame shape=(2636, 16)


,_id,createdAt,updatedAt,name,exposure,model,sensors,description,image,lastMeasurementAt,grouptag,loc,currentLocation_coordinates,currentLocation_type,currentLocation_timestamp,weblink
0,578207d56fea661300861f3b,2022-03-30T11:25:43.223Z,2026-11-18T16:27:47.580Z,emp-wetter,outdoor,homeEthernet,"[{'_id': '578207d56fea661300861f3d', 'title': 'UV-Intensität', 'unit': 'μW/cm²', 'sensorType': 'VEML6070', 'lastMeasurement': {'createdAt': '2026-11-18T16:27:47.560Z', 'value': '0'}}, {'_id': '578207d56fea661300861f3e', 'title': 'Beleuchtungsstärke', 'unit': 'lx', 'sensorType': 'TSL45315', 'lastMeasurement': {'createdAt': '2026-11-18T16:27:47.560Z', 'value': '0'}}, {'_id': '578207d56fea661300861f3f', 'title': 'Luftdruck', 'unit': 'hPa', 'sensorType': 'BMP280', 'lastMeasurement': {'createdAt': '2026-11-18T16:27:47.560Z', 'value': '975'}}, {'_id': '578207d56fea661300861f40', 'title': 'rel. Luftfeuchte', 'unit': '%', 'sensorType': 'HDC1008', 'lastMeasurement': {'createdAt': '2026-11-18T16:27:47.560Z', 'value': '95.98'}}, {'_id': '578207d56fea661300861f41', 'title': 'Temperatur', 'unit': '°C', 'sensorType': 'HDC1008', 'lastMeasurement': {'createdAt': '2026-11-18T16:27:47.560Z', 'value': '8.14'}}, {'_id': '57a990b9d46eb21200e231c4', 'sensorType': 'FWG14-V', 'unit': 'm/s', 'title': 'Windgeschwindigkeit', 'lastMeasurement': {'createdAt': '2026-11-18T16:27:47.560Z', 'value': '0.685'}}, {'_id': '57ab3d3791673313004b003d', 'sensorType': 'FWG14-D', 'unit': 'lx', 'title': 'Dämmerung', 'lastMeasurement': {'createdAt': '2026-11-18T16:27:47.560Z', 'value': '0'}, 'icon': 'osem-windspeed'}, {'_id': '57ab3d3791673313004b003e', 'sensorType': 'FWG14-R', 'unit': 'mm /m² Tag', 'title': 'Regen', 'lastMeasurement': {'createdAt': '2026-11-18T16:27:47.560Z', 'value': '7.711'}, 'icon': 'osem-sprinkles'}, {'_id': '57ab3d3791673313004b003f', 'sensorType': 'FWG14-SO', 'unit': 'kLx', 'title': 'Licht Sonne Ost', 'lastMeasurement': {'createdAt': '2026-11-18T16:27:47.560Z', 'value': '0'}, 'icon': 'osem-brightness'}, {'_id': '57ab3d3791673313004b0040', 'sensorType': 'FWG14-SS', 'unit': 'kLx', 'title': 'Licht Sonne Süd', 'lastMeasurement': {'createdAt': '2026-11-18T16:27:47.560Z', 'value': '0'}, 'icon': 'osem-brightness'}, {'_id': '57ab3d3791673313004b0041', 'sensorType': 'FWG14-SW', 'unit': 'kLx', 'title': 'Licht Sonne West', 'lastMeasurement': {'createdAt': '2026-11-18T16:27:47.560Z', 'value': '0'}, 'icon': 'osem-brightness'}, {'_id': '57b601ddb17d681100196369', 'sensorType': 'FWG14-RI', 'unit': '°', 'title': 'Windrichtung', 'lastMeasurement': {'createdAt': '2026-11-18T16:27:47.560Z', 'value': '292.5'}}, {'_id': '57dc18b72fc31e1600a7ba37', 'sensorType': 'SM10E101', 'unit': 'Wh', 'title': 'TagesLeistung', 'lastMeasurement': {'createdAt': '2026-11-18T16:27:47.560Z', 'value': '0'}}, {'_id': '57e426728859c00011179c6f', 'sensorType': 'SM10E102', 'unit': 'W', 'title': 'LeistungAktuell', 'lastMeasurement': {'createdAt': '2026-11-18T16:27:47.560Z', 'value': '3.81178'}, 'icon': 'osem-brightness'}, {'_id': '57e7ff818859c0001124419d', 'title': 'Grundwasserstand', 'unit': 'cm', 'sensorType': 'Pegelsonde', 'icon': 'osem-windspeed', 'lastMeasurement': {'createdAt': '2026-11-18T16:27:47.560Z', 'value': '139.383'}}, {'_id': '590034635ba9e50011e6f265', 'icon': 'osem-particulate-matter', 'sensorType': 'UP30A001', 'unit': 'µg/m³', 'title': 'Feinstaub PM2,5', 'lastMeasurement': {'createdAt': '2026-11-18T16:27:47.560Z', 'value': '0.6'}}, {'_id': '590034635ba9e50011e6f266', 'icon': 'osem-particulate-matter', 'sensorType': 'UP30A002', 'unit': 'µg/m³', 'title': 'Feinstaub PM10', 'lastMeasurement': {'value': '11.7', 'createdAt': '2026-11-18T07:25:33.259Z'}}]",Aufbereitung der Messwerte erfolgt durch einen Arduino Uno.\nÜber eine Modbus TCP-Schnittstelle werden die Messwert an eine Mini SPS übertragen. Diese SPS überträgt dann diese und weitere Daten als JSON-Array im 60s Takt.,578207d56fea661300861f3b_ovyk87.jpg,2026-11-18T16:27:47.560Z,[],"[{'geometry': {'coordinates': [9.391277, 51.660264], 't

In [16]:
def transform_boxes(boxes_df: pd.DataFrame) -> pd.DataFrame:
    """
    Return the outer box dataset without the nested sensors column.

    Preserves all remaining columns and their original names.
    Does not modify the input DataFrame.
    """
    if not isinstance(boxes_df, pd.DataFrame):
        raise TypeError("boxes_df must be a pandas DataFrame.")

    return boxes_df.drop(columns=["sensors"], errors="ignore").copy()

In [17]:
import pandas as pd


def transform_sensors(raw_boxes: pd.DataFrame):
    """
    Build a sensor dimension from the nested sensors column.

    Returns:
        sensors_df            -> unique sensors only
        duplicate_sensors_df  -> all duplicated sensor records for investigation
    """
    if not isinstance(raw_boxes, pd.DataFrame):
        raise TypeError("raw_boxes must be a pandas DataFrame.")

    required_columns = ["_id", "sensors"]

    missing = [col for col in required_columns if col not in raw_boxes.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    sensor_records = []

    # Only use the required outer columns.
    for _, box in raw_boxes[required_columns].iterrows():

        box_id = box["_id"]
        sensors = box["sensors"]

        if not isinstance(sensors, list):
            continue

        for sensor in sensors:

            if not isinstance(sensor, dict):
                continue

            # Preserve every field from the sensor dictionary.
            record = sensor.copy()

            # Standardize the sensor identifier.It removes the old key
            # _id and assigns a new key sensor_id to the same value in _id
            record["sensor_id"] = record.pop("_id", None)

            # Preserve the one-to-many relationship.
            record["box_id"] = box_id

            sensor_records.append(record)

    sensors_df = pd.DataFrame(sensor_records)

    # Keep identifiers first.
    priority = ["sensor_id", "box_id"]
    remaining = [c for c in sensors_df.columns if c not in priority]
    sensors_df = sensors_df[priority + remaining]

    # Identify duplicate sensor IDs.
    duplicate_mask = sensors_df["sensor_id"].duplicated(keep=False)

    duplicate_sensors_df = (
        sensors_df.loc[duplicate_mask]
        .sort_values(["sensor_id", "box_id"])
        .reset_index(drop=True)
    )

    # Keep only unique sensor IDs.
    sensors_df = (
        sensors_df.loc[~duplicate_mask]
        .reset_index(drop=True)
    )

    # Summary.
    print(f"Total extracted sensors : {len(sensor_records)}")
    print(f"Unique sensors retained : {len(sensors_df)}")
    print(f"Duplicate sensor records: {len(duplicate_sensors_df)}")

    if not duplicate_sensors_df.empty:
        print("\nDuplicate sensor IDs found:")
        display(duplicate_sensors_df)

    return sensors_df, duplicate_sensors_df

In [18]:
boxes_test_df = transform_boxes(preview_boxes_df)

In [19]:
boxes_test_df.head()

,_id,createdAt,updatedAt,name,exposure,model,description,image,lastMeasurementAt,grouptag,loc,currentLocation_coordinates,currentLocation_type,currentLocation_timestamp,weblink
0,578207d56fea661300861f3b,2022-03-30T11:25:43.223Z,2026-11-18T16:27:47.580Z,emp-wetter,outdoor,homeEthernet,Aufbereitung der Messwerte erfolgt durch einen Arduino Uno.\nÜber eine Modbus TCP-Schnittstelle werden die Messwert an eine Mini SPS übertragen. Diese SPS überträgt dann diese und weitere Daten als JSON-Array im 60s Takt.,578207d56fea661300861f3b_ovyk87.jpg,2026-11-18T16:27:47.560Z,[],"[{'geometry': {'coordinates': [9.391277, 51.660264], 'type': 'Point', 'timestamp': '2016-07-10T08:31:17.000Z'}, 'type': 'Feature'}]","[9.391277, 51.660264]",Point,2016-07-10T08:31:17.000Z,NaN
1,579e683668b4a21200661a6d,2022-03-30T11:25:43.225Z,2026-11-18T16:28:31.953Z,Lengfeld Hintergasse 11,outdoor,homeWifi,"Die Wetterstation steht in Otzberg, Ortsteil Lengfeld. Die senseBox basiert auf einem Arduino UNO Microcontroller-Board.",NaN,2026-11-18T16:28:31.943Z,[],"[{'geometry': {'coordinates': [8.90362, 49.832135], 'type': 'Point', 'timestamp': '2016-07-31T21:05:58.000Z'}, 'type': 'Feature'}]","[8.90362, 49.832135]",Point,2016-07-31T21:05:58.000Z,NaN
2,57aa3052f52f21100029a5a9,2016-08-09T19:34:42.077Z,2026-11-18T16:28:25.391Z,OBS Bohmte UK_02,outdoor,homeWifi,"Oberschule Bohmte, Umweltkataster, Sensorbox_02",NaN,2026-11-18T16:28:25.384Z,NaN,"[{'geometry': {'coordinates': [8.311629, 52.369553], 'type': 'Point', 'timestamp': '2016-08-09T19:34:42.000Z'}, 'type': 'Feature'}]","[8.311629, 52.369553]",Point,2016-08-09T19:34:42.000Z,NaN
3,580771bacd79a20011b5c0cd,2022-03-30T11:25:43.232Z,2026-11-18T16:26:28.094Z,Neu Zippendorf,outdoor,custom,"am Rande von Neu Zippendorf - Stadtteil von Schwerin.......\nBME280: Druck,Temperatur und Luftfeucht / TSL_TSL45315: Licht Lux / VEML 6070 UV-a",580771bacd79a20011b5c0cd.jpg?1488892556367,2026-11-18T16:26:28.080Z,[],"[{'geometry': {'coordinates': [11.446091, 53.596166], 'type': 'Point', 'timestamp': '2016-10-19T13:14:34.000Z'}, 'type': 'Feature'}]","[11.446091, 53.596166]",Point,2016-10-19T13:14:34.000Z,NaN
4,580f30787ac61b0010983265,2022-03-30T11:25:43.233Z,2026-11-18T16:28:13.616Z,KlimaBox_Helmstedt,indoor,homeEthernetFeinstaub,"Leider nur ""In-House""",580f30787ac61b0010983265.jpg?1487859095939,2026-11-18T16:28:13.601Z,NaN,"[{'geometry': {'type': 'Point', 'coordinates': [10.999567, 52.220761], 'timestamp': '2024-10-27T12:02:55.184Z'}, 'type': 'Feature'}]","[10.999567, 52.220761]",Point,2024-10-27T12:02:55.184Z,NaN


In [20]:
sensors_df_test, duplicate_sensors_df = transform_sensors(preview_boxes_df)

Total extracted sensors : 13596
Unique sensors retained : 13584
Duplicate sensor records: 12

Duplicate sensor IDs found:


,sensor_id,box_id,title,unit,sensorType,lastMeasurement,icon
0,595b27e794f0520011603112,584e9e9e16c08e000fb7e146,PM10,µg/m³,SDS 011,"{'value': '25.67', 'createdAt': '2026-11-18T07:24:07.814Z'}",osem-cloud
1,595b27e794f0520011603112,595b27e794f0520011603111,PM10,µg/m³,SDS 011,"{'createdAt': '2026-11-18T16:27:12.297Z', 'value': '15.15'}",osem-cloud
2,595b27e794f0520011603113,584e9e9e16c08e000fb7e146,PM2.5,µg/m³,SDS 011,None,osem-cloud
3,595b27e794f0520011603113,595b27e794f0520011603111,PM2.5,µg/m³,SDS 011,"{'value': '9.00', 'createdAt': '2026-11-18T07:24:07.814Z'}",osem-cloud
4,5968c9c41900bf0011fd792e,5968c9c41900bf0011fd792d,PM10,µg/m³,SDS 011,"{'value': '88.20', 'createdAt': '2026-11-18T07:24:50.094Z'}",osem-cloud
5,5968c9c41900bf0011fd792e,596914641900bf001101bd2c,PM10,µg/m³,SDS 011,"{'createdAt': '2026-11-18T16:27:17.310Z', 'value': '86.80'}",osem-cloud
6,5968c9c41900bf0011fd792f,5968c9c41900bf0011fd792d,PM2.5,µg/m³,SDS 011,"{'createdAt': '2026-10-17T11:00:12.514Z', 'value': '42.33'}",osem-cloud
7,5968c9c41900bf0011fd792f,596914641900bf001101bd2c,PM2.5,µg/m³,SDS 011,"{'value': '2.50', 'createdAt': '2026-11-18T07:24:50.094Z'}",osem-cloud
8,5970f5672e9396000fd6e0a9,592a682751d3460011d4562e,PM10,µg/m³,SDS 011,NaN,osem-cloud
9,5970f5672e9396000fd6e0a9,5970f5672e9396000fd6e0a8,PM10,µg/m³,SDS 011,"{'createdAt': '2026-11-18T16:27:07.574Z', 'value': '22.40'}",osem-cloud


In [21]:
sensors_df_test.head()

,sensor_id,box_id,title,unit,sensorType,lastMeasurement,icon
0,578207d56fea661300861f3d,578207d56fea661300861f3b,UV-Intensität,μW/cm²,VEML6070,"{'createdAt': '2026-11-18T16:27:47.560Z', 'value': '0'}",NaN
1,578207d56fea661300861f3e,578207d56fea661300861f3b,Beleuchtungsstärke,lx,TSL45315,"{'createdAt': '2026-11-18T16:27:47.560Z', 'value': '0'}",NaN
2,578207d56fea661300861f3f,578207d56fea661300861f3b,Luftdruck,hPa,BMP280,"{'createdAt': '2026-11-18T16:27:47.560Z', 'value': '975'}",NaN
3,578207d56fea661300861f40,578207d56fea661300861f3b,rel. Luftfeuchte,%,HDC1008,"{'createdAt': '2026-11-18T16:27:47.560Z', 'value': '95.98'}",NaN
4,578207d56fea661300861f41,578207d56fea661300861f3b,Temperatur,°C,HDC1008,"{'createdAt': '2026-11-18T16:27:47.560Z', 'value': '8.14'}",NaN


In [22]:
sensors_df_test['title'].value_counts()

title
Temperatur                      2686
rel. Luftfeuchte                2383
PM10                            2375
PM2.5                           2363
Luftdruck                       1811
                                ... 
Globalstrahlung                    1
PM5.0                              1
Niederschlag - aktuell             1
Niederschlag - letzte Stunde       1
Niederschlag - heute               1
Name: count, Length: 446, dtype: int64

In [23]:
preview_boxes_df["sensors"].value_counts()

sensors
[{'_id': '578207d56fea661300861f3d', 'title': 'UV-Intensität', 'unit': 'μW/cm²', 'sensorType': 'VEML6070', 'lastMeasurement': {'createdAt': '2026-11-18T16:27:47.560Z', 'value': '0'}}, {'_id': '578207d56fea661300861f3e', 'title': 'Beleuchtungsstärke', 'unit': 'lx', 'sensorType': 'TSL45315', 'lastMeasurement': {'createdAt': '2026-11-18T16:27:47.560Z', 'value': '0'}}, {'_id': '578207d56fea661300861f3f', 'title': 'Luftdruck', 'unit': 'hPa', 'sensorType': 'BMP280', 'lastMeasurement': {'createdAt': '2026-11-18T16:27:47.560Z', 'value': '975'}}, {'_id': '578207d56fea661300861f40', 'title': 'rel. Luftfeuchte', 'unit': '%', 'sensorType': 'HDC1008', 'lastMeasurement': {'createdAt': '2026-11-18T16:27:47.560Z', 'value': '95.98'}}, {'_id': '578207d56fea661300861f41', 'title': 'Temperatur', 'unit': '°C', 'sensorType': 'HDC1008', 'lastMeasurement': {'createdAt': '2026-11-18T16:27:47.560Z', 'value': '8.14'}}, {'_id': '57a990b9d46eb21200e231c4', 'sensorType': 'FWG14-V', 'unit': 'm/s', 'title': '

In [24]:
# Canonical output schema. These fields map the actual box + nested sensor + measurement payload.
# Complex API fields (grouptag and loc) are serialized as JSON strings to keep Parquet schema stable.
API_TO_EN = {
    "_id": "box_id",
    "createdAt": "measurement_created_at",
    "updatedAt": "measurement_updated_at",
    "name": "device_name",
    "exposure": "exposure",
    "model": "model",
    "sensors":"sensors",
    "description": "description",
    "image": "image",
    "lastMeasurementAt": "last_measurement_at",
    "grouptag": "grouptag",
    "loc": "location_geojson",
    "currentLocation_coordinates":"current_location",
    "currentLocation_type":"location_type",
    "currentLocation_timestamp":"timestamp",
    "weblink": "weblink",
    "sensorType": "sensor_type",
    "lastMeasurement":"last_measurement",
    
}
EN_TO_API = {v: k for k, v in API_TO_EN.items()}


In [25]:
def rename_to_english(df : pd.DataFrame) -> pd.DataFrame:
    """Rename API Column names to English schema names."""
    if df.empty:
        return df.copy()
    columns_to_rename = {}
    for src, dst in API_TO_EN.items():
        if src in df.columns and dst not in df.columns:
            columns_to_rename[src] = dst
    return df.rename(columns = columns_to_rename)

def rename_to_api_schema (df : pd.DataFrame) -> pd.DataFrame:
    """Rename English Column names back to API Schema Names(defensive / reverse operation)"""
    if df.empty:
        return df.copy()
    columns_to_rename = { k : v for k, v in EN_TO_API.items() if k in df.columns}
    return df.rename(columns=columns_to_rename)

In [26]:
english_boxes = rename_to_english(boxes_test_df)

In [27]:
english_sensors = rename_to_english(sensors_df_test)

In [28]:
english_boxes.columns

Index(['box_id', 'measurement_created_at', 'measurement_updated_at',
       'device_name', 'exposure', 'model', 'description', 'image',
       'last_measurement_at', 'grouptag', 'location_geojson',
       'current_location', 'location_type', 'timestamp', 'weblink'],
      dtype='str')

In [29]:
english_sensors.columns

Index(['sensor_id', 'box_id', 'title', 'unit', 'sensor_type',
       'last_measurement', 'icon'],
      dtype='str')

In [30]:
import pandas as pd
from typing import List, Optional

def flatten_dataframe(
    df: pd.DataFrame,
    nested_columns: Optional[List[str]] = None,
    sep: str = "_"
) -> pd.DataFrame:
    """
    Flatten selected columns containing nested dictionaries.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame.
    nested_columns : list[str], optional
        Columns containing nested dictionaries.
        If None, automatically detect columns containing dictionaries.
    sep : str
        Separator used when creating flattened column names.

    Returns
    -------
    pd.DataFrame
        Flattened copy of the input DataFrame.
    """
    df = df.copy()

    if nested_columns is None:
        nested_columns = [
            col for col in df.columns
            if df[col].map(
                lambda value: isinstance(value, dict)
            ).any()
        ]

    for col in nested_columns:
        if col not in df.columns:
            continue

        # Expand nested dictionary fields into separate columns
        expanded = pd.json_normalize(df[col])

        # Prefix new columns to preserve their origin
        expanded.columns = [
            f"{col}{sep}{subcol}"
            for subcol in expanded.columns
        ]

        # Preserve the original row index
        expanded.index = df.index

        # Remove the original nested column and attach its fields
        df = df.drop(columns=[col]).join(expanded)

    return df

In [31]:
sensors_flat_df = flatten_dataframe(
    english_sensors,
    nested_columns=["last_measurement"]
)

sensors_flat_df.head()

,sensor_id,box_id,title,unit,sensor_type,icon,last_measurement_createdAt,last_measurement_value
0,578207d56fea661300861f3d,578207d56fea661300861f3b,UV-Intensität,μW/cm²,VEML6070,NaN,2026-11-18T16:27:47.560Z,0
1,578207d56fea661300861f3e,578207d56fea661300861f3b,Beleuchtungsstärke,lx,TSL45315,NaN,2026-11-18T16:27:47.560Z,0
2,578207d56fea661300861f3f,578207d56fea661300861f3b,Luftdruck,hPa,BMP280,NaN,2026-11-18T16:27:47.560Z,975
3,578207d56fea661300861f40,578207d56fea661300861f3b,rel. Luftfeuchte,%,HDC1008,NaN,2026-11-18T16:27:47.560Z,95.98
4,578207d56fea661300861f41,578207d56fea661300861f3b,Temperatur,°C,HDC1008,NaN,2026-11-18T16:27:47.560Z,8.14


In [32]:
english_boxes.columns

Index(['box_id', 'measurement_created_at', 'measurement_updated_at',
       'device_name', 'exposure', 'model', 'description', 'image',
       'last_measurement_at', 'grouptag', 'location_geojson',
       'current_location', 'location_type', 'timestamp', 'weblink'],
      dtype='str')

In [33]:
sensors_flat_df.columns

Index(['sensor_id', 'box_id', 'title', 'unit', 'sensor_type', 'icon',
       'last_measurement_createdAt', 'last_measurement_value'],
      dtype='str')

In [34]:
english_boxes.head()

,box_id,measurement_created_at,measurement_updated_at,device_name,exposure,model,description,image,last_measurement_at,grouptag,location_geojson,current_location,location_type,timestamp,weblink
0,578207d56fea661300861f3b,2022-03-30T11:25:43.223Z,2026-11-18T16:27:47.580Z,emp-wetter,outdoor,homeEthernet,Aufbereitung der Messwerte erfolgt durch einen Arduino Uno.\nÜber eine Modbus TCP-Schnittstelle werden die Messwert an eine Mini SPS übertragen. Diese SPS überträgt dann diese und weitere Daten als JSON-Array im 60s Takt.,578207d56fea661300861f3b_ovyk87.jpg,2026-11-18T16:27:47.560Z,[],"[{'geometry': {'coordinates': [9.391277, 51.660264], 'type': 'Point', 'timestamp': '2016-07-10T08:31:17.000Z'}, 'type': 'Feature'}]","[9.391277, 51.660264]",Point,2016-07-10T08:31:17.000Z,NaN
1,579e683668b4a21200661a6d,2022-03-30T11:25:43.225Z,2026-11-18T16:28:31.953Z,Lengfeld Hintergasse 11,outdoor,homeWifi,"Die Wetterstation steht in Otzberg, Ortsteil Lengfeld. Die senseBox basiert auf einem Arduino UNO Microcontroller-Board.",NaN,2026-11-18T16:28:31.943Z,[],"[{'geometry': {'coordinates': [8.90362, 49.832135], 'type': 'Point', 'timestamp': '2016-07-31T21:05:58.000Z'}, 'type': 'Feature'}]","[8.90362, 49.832135]",Point,2016-07-31T21:05:58.000Z,NaN
2,57aa3052f52f21100029a5a9,2016-08-09T19:34:42.077Z,2026-11-18T16:28:25.391Z,OBS Bohmte UK_02,outdoor,homeWifi,"Oberschule Bohmte, Umweltkataster, Sensorbox_02",NaN,2026-11-18T16:28:25.384Z,NaN,"[{'geometry': {'coordinates': [8.311629, 52.369553], 'type': 'Point', 'timestamp': '2016-08-09T19:34:42.000Z'}, 'type': 'Feature'}]","[8.311629, 52.369553]",Point,2016-08-09T19:34:42.000Z,NaN
3,580771bacd79a20011b5c0cd,2022-03-30T11:25:43.232Z,2026-11-18T16:26:28.094Z,Neu Zippendorf,outdoor,custom,"am Rande von Neu Zippendorf - Stadtteil von Schwerin.......\nBME280: Druck,Temperatur und Luftfeucht / TSL_TSL45315: Licht Lux / VEML 6070 UV-a",580771bacd79a20011b5c0cd.jpg?1488892556367,2026-11-18T16:26:28.080Z,[],"[{'geometry': {'coordinates': [11.446091, 53.596166], 'type': 'Point', 'timestamp': '2016-10-19T13:14:34.000Z'}, 'type': 'Feature'}]","[11.446091, 53.596166]",Point,2016-10-19T13:14:34.000Z,NaN
4,580f30787ac61b0010983265,2022-03-30T11:25:43.233Z,2026-11-18T16:28:13.616Z,KlimaBox_Helmstedt,indoor,homeEthernetFeinstaub,"Leider nur ""In-House""",580f30787ac61b0010983265.jpg?1487859095939,2026-11-18T16:28:13.601Z,NaN,"[{'geometry': {'type': 'Point', 'coordinates': [10.999567, 52.220761], 'timestamp': '2024-10-27T12:02:55.184Z'}, 'type': 'Feature'}]","[10.999567, 52.220761]",Point,2024-10-27T12:02:55.184Z,NaN


In [35]:
import pandas as pd


def prepare_location_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare nested location data for flattening.

    - Extracts the first GeoJSON Feature from location_geojson.
    - Converts current_location coordinates into longitude/latitude fields.
    - Preserves the original DataFrame by working on a copy.
    - Handles missing or unexpected values safely.
    """
    df = df.copy()

    # Prepare the GeoJSON Feature
    if "location_geojson" in df.columns:
        df["location_geojson"] = df["location_geojson"].apply(
            lambda value: (
                value[0]
                if isinstance(value, list) and value
                else None
            )
        )

    # Prepare the coordinates
    if "current_location" in df.columns:
        df["current_location"] = df["current_location"].apply(
            lambda value: (
                {
                    "longitude": value[0],
                    "latitude": value[1]
                }
                if isinstance(value, (list, tuple))
                and len(value) >= 2
                else None
            )
        )

    return df

In [36]:
# Step 1: Prepare nested location structures
boxes_location_df = prepare_location_columns(english_boxes)

# Step 2: Flatten the prepared dictionaries
boxes_location_df = flatten_dataframe(
    boxes_location_df,
    nested_columns=[
        "location_geojson",
        "current_location"
    ]
)

# Step 3: Inspect the results
print(boxes_location_df.columns.tolist())
display(boxes_location_df.head())

['box_id', 'measurement_created_at', 'measurement_updated_at', 'device_name', 'exposure', 'model', 'description', 'image', 'last_measurement_at', 'grouptag', 'location_type', 'timestamp', 'weblink', 'location_geojson_type', 'location_geojson_geometry.coordinates', 'location_geojson_geometry.type', 'location_geojson_geometry.timestamp', 'current_location_longitude', 'current_location_latitude']


,box_id,measurement_created_at,measurement_updated_at,device_name,exposure,model,description,image,last_measurement_at,grouptag,location_type,timestamp,weblink,location_geojson_type,location_geojson_geometry.coordinates,location_geojson_geometry.type,location_geojson_geometry.timestamp,current_location_longitude,current_location_latitude
0,578207d56fea661300861f3b,2022-03-30T11:25:43.223Z,2026-11-18T16:27:47.580Z,emp-wetter,outdoor,homeEthernet,Aufbereitung der Messwerte erfolgt durch einen Arduino Uno.\nÜber eine Modbus TCP-Schnittstelle werden die Messwert an eine Mini SPS übertragen. Diese SPS überträgt dann diese und weitere Daten als JSON-Array im 60s Takt.,578207d56fea661300861f3b_ovyk87.jpg,2026-11-18T16:27:47.560Z,[],Point,2016-07-10T08:31:17.000Z,NaN,Feature,"[9.391277, 51.660264]",Point,2016-07-10T08:31:17.000Z,9.391277,51.660264
1,579e683668b4a21200661a6d,2022-03-30T11:25:43.225Z,2026-11-18T16:28:31.953Z,Lengfeld Hintergasse 11,outdoor,homeWifi,"Die Wetterstation steht in Otzberg, Ortsteil Lengfeld. Die senseBox basiert auf einem Arduino UNO Microcontroller-Board.",NaN,2026-11-18T16:28:31.943Z,[],Point,2016-07-31T21:05:58.000Z,NaN,Feature,"[8.90362, 49.832135]",Point,2016-07-31T21:05:58.000Z,8.903620,49.832135
2,57aa3052f52f21100029a5a9,2016-08-09T19:34:42.077Z,2026-11-18T16:28:25.391Z,OBS Bohmte UK_02,outdoor,homeWifi,"Oberschule Bohmte, Umweltkataster, Sensorbox_02",NaN,2026-11-18T16:28:25.384Z,NaN,Point,2016-08-09T19:34:42.000Z,NaN,Feature,"[8.311629, 52.369553]",Point,2016-08-09T19:34:42.000Z,8.311629,52.369553
3,580771bacd79a20011b5c0cd,2022-03-30T11:25:43.232Z,2026-11-18T16:26:28.094Z,Neu Zippendorf,outdoor,custom,"am Rande von Neu Zippendorf - Stadtteil von Schwerin.......\nBME280: Druck,Temperatur und Luftfeucht / TSL_TSL45315: Licht Lux / VEML 6070 UV-a",580771bacd79a20011b5c0cd.jpg?1488892556367,2026-11-18T16:26:28.080Z,[],Point,2016-10-19T13:14:34.000Z,NaN,Feature,"[11.446091, 53.596166]",Point,2016-10-19T13:14:34.000Z,11.446091,53.596166
4,580f30787ac61b0010983265,2022-03-30T11:25:43.233Z,2026-11-18T16:28:13.616Z,KlimaBox_Helmstedt,indoor,homeEthernetFeinstaub,"Leider nur ""In-House""",580f30787ac61b0010983265.jpg?1487859095939,2026-11-18T16:28:13.601Z,NaN,Point,2024-10-27T12:02:55.184Z,NaN,Feature,"[10.999567, 52.220761]",Point,2024-10-27T12:02:55.184Z,10.999567,52.220761


In [37]:
boxes_location_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2636 entries, 0 to 2635
Data columns (total 19 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   box_id                                 2636 non-null   str    
 1   measurement_created_at                 2636 non-null   str    
 2   measurement_updated_at                 2636 non-null   str    
 3   device_name                            2636 non-null   str    
 4   exposure                               2636 non-null   str    
 5   model                                  2636 non-null   str    
 6   description                            856 non-null    str    
 7   image                                  444 non-null    str    
 8   last_measurement_at                    2636 non-null   str    
 9   grouptag                               2123 non-null   object 
 10  location_type                          2636 non-null   str    
 11  timestamp      

In [38]:
sensors_flat_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13584 entries, 0 to 13583
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   sensor_id                   13584 non-null  str  
 1   box_id                      13584 non-null  str  
 2   title                       13584 non-null  str  
 3   unit                        13584 non-null  str  
 4   sensor_type                 13570 non-null  str  
 5   icon                        13554 non-null  str  
 6   last_measurement_createdAt  13017 non-null  str  
 7   last_measurement_value      13017 non-null  str  
dtypes: str(8)
memory usage: 2.2 MB


In [39]:
from typing import List
import pandas as pd


def convert_to_date(
    df: pd.DataFrame,
    columns: List[str]
) -> pd.DataFrame:
    """
    Convert specified IoT timestamp columns to datetime.

    Missing or invalid timestamps become NaT.
    Timestamps are parsed as UTC and retain their time precision.
    """
    df = df.copy()

    for col in columns:
        if col in df.columns:
            df[col] = pd.to_datetime(
                df[col],
                errors="coerce",
                utc=True
            )

    return df


def convert_to_numeric(
    df: pd.DataFrame,
    columns: List[str]
) -> pd.DataFrame:
    """
    Convert specified IoT columns to numeric types.

    Invalid or missing values become NaN.
    IDs and other columns remain unchanged.
    """
    df = df.copy()

    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

    return df



In [40]:
# Timestamp columns in the boxes DataFrame
box_date_columns = [
    "measurement_created_at",
    "measurement_updated_at",
    "last_measurement_at",
    "timestamp",
    "location_geojson_geometry.timestamp",
]

# Timestamp columns in the sensors DataFrame
sensor_date_columns = [
    "last_measurement_createdAt",
]

# Numeric columns in the boxes DataFrame
box_numeric_columns = [
    "current_location_longitude",
    "current_location_latitude",
]

# Numeric columns in the sensors DataFrame
sensor_numeric_columns = [
    "last_measurement_value",
]

# Transform the boxes DataFrame
boxes_location_df = convert_to_date(
    boxes_location_df,
    box_date_columns
)

boxes_location_df = convert_to_numeric(
    boxes_location_df,
    box_numeric_columns
)

# Transform the sensors DataFrame
sensors_flat_df = convert_to_date(
    sensors_flat_df,
    sensor_date_columns
)

sensors_flat_df = convert_to_numeric(
    sensors_flat_df,
    sensor_numeric_columns
)

# Inspect the resulting data types
print("BOX DATA TYPES")
print(boxes_location_df.dtypes)

print("\nSENSOR DATA TYPES")
print(sensors_flat_df.dtypes)


BOX DATA TYPES
box_id                                                   str
measurement_created_at                   datetime64[us, UTC]
measurement_updated_at                   datetime64[us, UTC]
device_name                                              str
exposure                                                 str
model                                                    str
description                                              str
image                                                    str
last_measurement_at                      datetime64[us, UTC]
grouptag                                              object
location_type                                            str
timestamp                                datetime64[us, UTC]
weblink                                                  str
location_geojson_type                                    str
location_geojson_geometry.coordinates                 object
location_geojson_geometry.type                           str
location_

In [41]:
boxes_location_df['grouptag'].value_counts()

grouptag
[]                          659
[]                          624
[Luftdaten]                  90
[#stropdeaer]                71
[CML werkgroep fijnstof]     48
                           ... 
[airroh]                      1
[CPIE]                        1
[vital2]                      1
[HES]                         1
[15497688]                    1
Name: count, Length: 324, dtype: int64

In [42]:
messy_df = pd.DataFrame({
    "measurement_time": ["2026-01-15T10:00:00Z", "2026-01-16T10:00:00Z", "not a date", None],
    "value": [21.5, "22.1", "N/A", None],
})
type_safe_df = convert_to_date(messy_df, ["measurement_time"])
type_safe_df = convert_to_numeric(type_safe_df, ["value"])
# The pipeline continues safely when sensor timestamps or readings are malformed.


In [43]:
from typing import List
import pandas as pd


def extract_time_features(
    df: pd.DataFrame,
    date_columns: List[str]
) -> pd.DataFrame:
    """
    Extract calendar and time features from datetime columns.

    Adds year, month, day, hour, minute, and day-of-week
    columns while preserving the original datetime columns.
    """
    df = df.copy()

    for col in date_columns:
        # Skip columns that do not exist
        if col not in df.columns:
            continue

        # Skip columns that are not datetime types
        if not pd.api.types.is_datetime64_any_dtype(df[col]):
            continue

        # Create a prefix based on the original column name
        prefix = col.replace(".", "_")

        df[f"{prefix}_year"] = df[col].dt.year
        df[f"{prefix}_month"] = df[col].dt.month
        df[f"{prefix}_day"] = df[col].dt.day
        df[f"{prefix}_hour"] = df[col].dt.hour
        df[f"{prefix}_minute"] = df[col].dt.minute
        df[f"{prefix}_day_of_week"] = df[col].dt.dayofweek

    return df


In [44]:
box_date_columns = [
    "measurement_created_at",
    "measurement_updated_at",
    "last_measurement_at",
    "timestamp",
    "location_geojson_geometry.timestamp",
]

sensor_date_columns = [
    "last_measurement_createdAt",
]

boxes_location_df = extract_time_features(
    boxes_location_df,
    box_date_columns
)

sensors_flat_df = extract_time_features(
    sensors_flat_df,
    sensor_date_columns
)


In [45]:
print("Box DataFrame shape:", boxes_location_df.shape)
print("Sensor DataFrame shape:", sensors_flat_df.shape)
print("Box DataFrame Summary:", boxes_location_df.info())
print("Sensor DataFrame Summary:", sensors_flat_df.info())

Box DataFrame shape: (2636, 49)
Sensor DataFrame shape: (13584, 14)
<class 'pandas.DataFrame'>
RangeIndex: 2636 entries, 0 to 2635
Data columns (total 49 columns):
 #   Column                                           Non-Null Count  Dtype              
---  ------                                           --------------  -----              
 0   box_id                                           2636 non-null   str                
 1   measurement_created_at                           2636 non-null   datetime64[us, UTC]
 2   measurement_updated_at                           2636 non-null   datetime64[us, UTC]
 3   device_name                                      2636 non-null   str                
 4   exposure                                         2636 non-null   str                
 5   model                                            2636 non-null   str                
 6   description                                      856 non-null    str                
 7   image                

In [46]:
boxes_location_df.head()

,box_id,measurement_created_at,measurement_updated_at,device_name,exposure,model,description,image,last_measurement_at,grouptag,location_type,timestamp,weblink,location_geojson_type,location_geojson_geometry.coordinates,location_geojson_geometry.type,location_geojson_geometry.timestamp,current_location_longitude,current_location_latitude,measurement_created_at_year,measurement_created_at_month,measurement_created_at_day,measurement_created_at_hour,measurement_created_at_minute,measurement_created_at_day_of_week,measurement_updated_at_year,measurement_updated_at_month,measurement_updated_at_day,measurement_updated_at_hour,measurement_updated_at_minute,measurement_updated_at_day_of_week,last_measurement_at_year,last_measurement_at_month,last_measurement_at_day,last_measurement_at_hour,last_measurement_at_minute,last_measurement_at_day_of_week,timestamp_year,timestamp_month,timestamp_day,timestamp_hour,timestamp_minute,timestamp_day_of_week,location_geojson_geometry_timestamp_year,location_geojson_geometry_timestamp_month,location_geojson_geometry_timestamp_day,location_geojson_geometry_timestamp_hour,location_geojson_geometry_timestamp_minute,location_geojson_geometry_timestamp_day_of_week
0,578207d56fea661300861f3b,2022-03-30 11:25:43.223000+00:00,2026-11-18 16:27:47.580000+00:00,emp-wetter,outdoor,homeEthernet,Aufbereitung der Messwerte erfolgt durch einen Arduino Uno.\nÜber eine Modbus TCP-Schnittstelle werden die Messwert an eine Mini SPS übertragen. Diese SPS überträgt dann diese und weitere Daten als JSON-Array im 60s Takt.,578207d56fea661300861f3b_ovyk87.jpg,2026-11-18 16:27:47.560000+00:00,[],Point,2016-07-10 08:31:17+00:00,NaN,Feature,"[9.391277, 51.660264]",Point,2016-07-10 08:31:17+00:00,9.391277,51.660264,2022,3,30,11,25,2,2026,11,18,16,27,2,2026,11,18,16,27,2,2016,7,10,8,31,6,2016,7,10,8,31,6
1,579e683668b4a21200661a6d,2022-03-30 11:25:43.225000+00:00,2026-11-18 16:28:31.953000+00:00,Lengfeld Hintergasse 11,outdoor,homeWifi,"Die Wetterstation steht in Otzberg, Ortsteil Lengfeld. Die senseBox basiert auf einem Arduino UNO Microcontroller-Board.",NaN,2026-11-18 16:28:31.943000+00:00,[],Point,2016-07-31 21:05:58+00:00,NaN,Feature,"[8.90362, 49.832135]",Point,2016-07-31 21:05:58+00:00,8.903620,49.832135,2022,3,30,11,25,2,2026,11,18,16,28,2,2026,11,18,16,28,2,2016,7,31,21,5,6,2016,7,31,21,5,6
2,57aa3052f52f21100029a5a9,2016-08-09 19:34:42.077000+00:00,2026-11-18 16:28:25.391000+00:00,OBS Bohmte UK_02,outdoor,homeWifi,"Oberschule Bohmte, Umweltkataster, Sensorbox_02",NaN,2026-11-18 16:28:25.384000+00:00,NaN,Point,2016-08-09 19:34:42+00:00,NaN,Feature,"[8.311629, 52.369553]",Point,2016-08-09 19:34:42+00:00,8.311629,52.369553,2016,8,9,19,34,1,2026,11,18,16,28,2,2026,11,18,16,28,2,2016,8,9,19,34,1,2016,8,9,19,34,1
3,580771bacd79a20011b5c0cd,2022-03-30 11:25:43.232000+00:00,2026-11-18 16:26:28.094000+00:00,Neu Zippendorf,outdoor,custom,"am Rande von Neu Zippendorf - Stadtteil von Schwerin.......\nBME280: Druck,Temperatur und Luftfeucht / TSL_TSL45315: Licht Lux / VEML 6070 UV-a",580771bacd79a20011b5c0cd.jpg?1488892556367,2026-11-18 16:26:28.080000+00:00,[],Point,2016-10-19 13:14:34+00:00,NaN,Feature,"[11.446091, 53.596166]",Point,2016-10-19 13:14:34+00:00,11.446091,53.596166,2022,3,30,11,25,2,2026,11,18,16,26,2,2026,11,18,16,26,2,2016,10,19,13,14,2,2016,10,19,13,14,2
4,580f30787ac61b0010983265,2022-03-30 11:25:43.233000+00:00,2026-11-18 16:28:13.616000+00:00,KlimaBox_Helmstedt,indoor,homeEthernetFeinstaub,"Leider nur ""In-House""",580f30787ac61b0010983265.jpg?1487859095939,2026-11-18 16:28:13.601000+00:00,NaN,Point,2024-10-27 12:02:55.184000+00:00,NaN,Feature,"[10.999567, 52.220761]",Point,2024-10-27 12:02:55.184000+00:00,10.999567,52.220761,2022,3,30,11,25,2,2026,11,18,16,28,2,2026,11,18,16,28,2,2024,10,27,12,2,6,2024,10,27,12,2,6


In [47]:
sensors_flat_df.head()

,sensor_id,box_id,title,unit,sensor_type,icon,last_measurement_createdAt,last_measurement_value,last_measurement_createdAt_year,last_measurement_createdAt_month,last_measurement_createdAt_day,last_measurement_createdAt_hour,last_measurement_createdAt_minute,last_measurement_createdAt_day_of_week
0,578207d56fea661300861f3d,578207d56fea661300861f3b,UV-Intensität,μW/cm²,VEML6070,NaN,2026-11-18 16:27:47.560000+00:00,0.00,2026.0,11.0,18.0,16.0,27.0,2.0
1,578207d56fea661300861f3e,578207d56fea661300861f3b,Beleuchtungsstärke,lx,TSL45315,NaN,2026-11-18 16:27:47.560000+00:00,0.00,2026.0,11.0,18.0,16.0,27.0,2.0
2,578207d56fea661300861f3f,578207d56fea661300861f3b,Luftdruck,hPa,BMP280,NaN,2026-11-18 16:27:47.560000+00:00,975.00,2026.0,11.0,18.0,16.0,27.0,2.0
3,578207d56fea661300861f40,578207d56fea661300861f3b,rel. Luftfeuchte,%,HDC1008,NaN,2026-11-18 16:27:47.560000+00:00,95.98,2026.0,11.0,18.0,16.0,27.0,2.0
4,578207d56fea661300861f41,578207d56fea661300861f3b,Temperatur,°C,HDC1008,NaN,2026-11-18 16:27:47.560000+00:00,8.14,2026.0,11.0,18.0,16.0,27.0,2.0


In [48]:
sensors_flat_df['last_measurement_createdAt_year'].value_counts()

last_measurement_createdAt_year
2026.0    12833
2025.0      119
2024.0       65
Name: count, dtype: int64

In [49]:
sensors_flat_df['last_measurement_createdAt_month'].value_counts()

last_measurement_createdAt_month
11.0    12449
9.0       132
10.0      120
7.0        61
6.0        56
8.0        55
5.0        46
12.0       27
4.0        26
1.0        22
3.0        13
2.0        10
Name: count, dtype: int64

In [50]:
def inspect_unique_values(column_data, column_name):
    """
    Print the number of unique values and the unique values
    in a Pandas Series.

    Parameters:
        column_data: DataFrame column, e.g. sensors_df["title"]
        column_name: Name of the column being inspected
    """
    unique_values = column_data.dropna().unique()

    print(f"\n{'=' * 60}")
    print(f"Column: {column_name}")
    print(f"Number of unique values: {len(unique_values)}")
    print("\nUnique values:")

    for value in unique_values:
        print(value)

In [51]:
# pip install transformers torch sentencepiece

In [53]:
import torch
from transformers import MarianMTModel, MarianTokenizer

MODEL_NAME = "Helsinki-NLP/opus-mt-de-en"

_tokenizer = None
_model = None

# Known sensor terminology: translate these without calling the model.
SENSOR_LABEL_MAP = {
    "Temperatur": "Temperature",
    "Luftdruck": "Air pressure",
    "Beleuchtungsstärke": "Illuminance",
    "UV-Intensität": "UV intensity",
    "rel. Luftfeuchte": "Relative humidity",
    "Feinstaub PM10": "Particulate matter PM10",
    "Feinstaub PM2.5": "Particulate matter PM2.5",
    "Niederschlag letzte Stunde": "Precipitation in the last hour",
    "Tagesniederschlag": "Daily precipitation",
}


def get_translation_model():
    """Load MarianMT once and reuse it for subsequent translations."""
    global _tokenizer, _model

    if _tokenizer is None or _model is None:
        _tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
        _model = MarianMTModel.from_pretrained(MODEL_NAME)

        _model.to("cpu")
        _model.eval()

        torch.set_num_threads(2)

    return _tokenizer, _model


def translate_iot_text(
    boxes_location_df: pd.DataFrame,
    sensors_flat_df: pd.DataFrame,
    batch_size: int = 4
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Translate German descriptive text in boxes_df and sensors_df into English.

    Preserves IDs, timestamps, coordinates, units, sensor types,
    numeric measurements and all other non-text fields.
    """

    boxes_df = boxes_location_df.copy()
    sensors_df = sensors_flat_df.copy()

    translation_cache = SENSOR_LABEL_MAP.copy()

    # Only translate descriptive text, not technical identifiers or measurements.
    box_text_columns = [
        col for col in [
            "name", "device_name", "exposure",
            "description", "grouptag"
        ]
        if col in boxes_df.columns
    ]

    sensor_text_columns = [
        col for col in ["title"]
        if col in sensors_df.columns
    ]

    # Collect unique text values that still need translation.
    text_values = set()

    for df, columns in [
        (boxes_df, box_text_columns),
        (sensors_df, sensor_text_columns)
    ]:
        for column in columns:
            values = df[column].dropna()

            text_values.update(
                value.strip()
                for value in values
                if isinstance(value, str) and value.strip()
            )

    pending_translations = sorted(
        value for value in text_values
        if value not in translation_cache
    )

    # Translate unknown German text in batches.
    if pending_translations:
        tokenizer, model = get_translation_model()

        for start in range(0, len(pending_translations), batch_size):
            batch = pending_translations[start:start + batch_size]

            inputs = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=256
            )

            with torch.inference_mode():
                translated_ids = model.generate(
                    **inputs,
                    max_length=256
                )

            translations = tokenizer.batch_decode(
                translated_ids,
                skip_special_tokens=True
            )

            translation_cache.update(
                dict(zip(batch, translations))
            )

    # Apply cached translations to both DataFrames.
    for df, columns in [
        (boxes_df, box_text_columns),
        (sensors_df, sensor_text_columns)
    ]:
        for column in columns:
            df[column] = df[column].apply(
                lambda value: (
                    translation_cache.get(value.strip(), value)
                    if isinstance(value, str)
                    else value
                )
            )

    print("German-to-English translation completed.")
    print("Box text columns translated:", box_text_columns)
    print("Sensor text columns translated:", sensor_text_columns)
    print("Unique text values processed:", len(text_values))

    return boxes_df, sensors_df

In [54]:
boxes_en, sensors_en = translate_iot_text(
    boxes_location_df=boxes_location_df,
    sensors_flat_df=sensors_flat_df,
    batch_size=4
)

/home/collonel/data-engineering/Retail_Data_Warehouse/.venv/lib/python3.14/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100%|███████████████████████████████████████████████████████████████| 258/258 [00:07<00:00, 34.58it/s]


German-to-English translation completed.
Box text columns translated: ['device_name', 'exposure', 'description', 'grouptag']
Sensor text columns translated: ['title']
Unique text values processed: 3773


In [55]:
print("BOXES")
print("Original shape:", boxes_location_df.shape)
print("Translated shape:", boxes_en.shape)

print("\nSENSORS")
print("Original shape:", sensors_flat_df.shape)
print("Translated shape:", sensors_en.shape)

print("\nTRANSLATED SENSOR TITLES")
display(
    sensors_en[["sensor_id", "title", "unit"]]
    .drop_duplicates()
    .head(20)
)

print("\nTRANSLATED BOX DESCRIPTIONS")
display(
    boxes_en[["box_id", "description"]]
    .dropna(subset=["description"])
    .head(10)
)

print("\nTRANSLATED GROUP TAGS")
display(
    boxes_en[["box_id", "grouptag"]]
    .dropna(subset=["grouptag"])
    .head(10)
)

BOXES
Original shape: (2636, 49)
Translated shape: (2636, 49)

SENSORS
Original shape: (13584, 14)
Translated shape: (13584, 14)

TRANSLATED SENSOR TITLES


,sensor_id,title,unit
0,578207d56fea661300861f3d,UV intensity,μW/cm²
1,578207d56fea661300861f3e,Illuminance,lx
2,578207d56fea661300861f3f,Air pressure,hPa
3,578207d56fea661300861f40,Relative humidity,%
4,578207d56fea661300861f41,Temperature,°C
5,57a990b9d46eb21200e231c4,Wind speed,m/s
6,57ab3d3791673313004b003d,Twilight,lx
7,57ab3d3791673313004b003e,Rain,mm /m² Tag
8,57ab3d3791673313004b003f,Light Sun East,kLx
9,57ab3d3791673313004b0040,Light Sun South,kLx



TRANSLATED BOX DESCRIPTIONS


,box_id,description
0,578207d56fea661300861f3b,"The measured values are processed by an Arduino Uno. Via a Modbus TCP interface, the measured value is transmitted to a Mini PLC. This PLC then transmits this and other data as a JSON array in the 60s clock."
1,579e683668b4a21200661a6d,"The weather station is located in Otzberg, Lengfeld. The senseBox is based on an Arduino UNO microcontroller board."
2,57aa3052f52f21100029a5a9,"Bohmte High School, Environmental Cataster, Sensorbox_02"
3,580771bacd79a20011b5c0cd,"on the edge of Neu Zippendorf - district of Schwerin....... BME280: Pressure,temperature and humidity / TSL_TSL45315: Light Lux / VEML 6070 UV-a"
4,580f30787ac61b0010983265,"Unfortunately, only ""in-house"""
5,581dab879337ed000f6b308d,Measurement data from the Rademacher environmental sensor / BME280 / particulate sensor
6,582473be40198a001010e1a3,"Sensebox on the balcony of the plant, fungal and soil ecology lab at the Freie Universität Berlin."
7,584e9e9e16c08e000fb7e146,City Hall City of Kerpen
8,5879ecfd0eb1a0000f36a145,Ecowitt Wittboy GW2001
9,5882335198a4760014d47963,weather data at the office



TRANSLATED GROUP TAGS


,box_id,grouptag
0,578207d56fea661300861f3b,[]
1,579e683668b4a21200661a6d,[]
3,580771bacd79a20011b5c0cd,[]
5,581dab879337ed000f6b308d,[]
6,582473be40198a001010e1a3,[MakeLight]
7,584e9e9e16c08e000fb7e146,[KDVZ]
8,5879ecfd0eb1a0000f36a145,[]
9,5882335198a4760014d47963,[]
10,5889cb92ff876d00101be9dd,[]
11,58d40641c877fb0011ac43e6,[]


In [56]:

def combine_iot_data(
    boxes_df: pd.DataFrame,
    sensors_df: pd.DataFrame
) -> pd.DataFrame:
    """
    Combine translated box and sensor DataFrames.

    Preserves sensor-level granularity:
    one row per sensor, with its corresponding box metadata.
    """

    required_box_columns = ["box_id"]
    required_sensor_columns = ["box_id", "sensor_id"]

    missing_boxes = [
        col for col in required_box_columns
        if col not in boxes_df.columns
    ]

    missing_sensors = [
        col for col in required_sensor_columns
        if col not in sensors_df.columns
    ]

    if missing_boxes or missing_sensors:
        raise KeyError(
            f"Missing box columns: {missing_boxes}; "
            f"missing sensor columns: {missing_sensors}"
        )

    if boxes_df["box_id"].isna().any():
        raise ValueError("Box DataFrame contains null box_id values.")

    if sensors_df["box_id"].isna().any():
        raise ValueError("Sensor DataFrame contains null box_id values.")

    if boxes_df["box_id"].duplicated().any():
        raise ValueError(
            "Box DataFrame contains duplicate box_id values. "
            "Resolve these before merging."
        )

    if sensors_df["sensor_id"].duplicated().any():
        raise ValueError(
            "Sensor DataFrame contains duplicate sensor_id values."
        )

    original_sensor_count = len(sensors_df)

    combined_df = sensors_df.merge(
        boxes_df,
        on="box_id",
        how="left",
        validate="many_to_one",
        suffixes=("_sensor", "_box"),
        indicator=True
    )

    # Check whether every sensor found its corresponding box.
    unmatched_count = (
        combined_df["_merge"] != "both"
    ).sum()

    print("MERGE SUMMARY")
    print("Original sensors:", original_sensor_count)
    print("Combined records:", len(combined_df))
    print("Matched sensors:", (combined_df["_merge"] == "both").sum())
    print("Unmatched sensors:", unmatched_count)

    # Remove the temporary merge indicator.
    combined_df = combined_df.drop(columns=["_merge"])

    # Ensure the merge did not multiply or lose sensor records.
    if len(combined_df) != original_sensor_count:
        raise ValueError(
            "Sensor record count changed during merge."
        )

    if combined_df["sensor_id"].duplicated().any():
        raise ValueError(
            "Duplicate sensor IDs detected after merging."
        )

    return combined_df.reset_index(drop=True)


In [57]:
iot_sensor_catalog_df = combine_iot_data(
    boxes_df=boxes_en,
    sensors_df=sensors_en
)

MERGE SUMMARY
Original sensors: 13584
Combined records: 13584
Matched sensors: 13584
Unmatched sensors: 0


In [58]:
print("Original boxes:", len(boxes_location_df))
print("Original sensors:", len(sensors_flat_df))
print("Translated boxes:", len(boxes_en))
print("Translated sensors:", len(sensors_en))
print("Final catalog:", len(iot_sensor_catalog_df))

print("\nFinal dataset shape:", iot_sensor_catalog_df.shape)

display(iot_sensor_catalog_df.head())

Original boxes: 2636
Original sensors: 13584
Translated boxes: 2636
Translated sensors: 13584
Final catalog: 13584

Final dataset shape: (13584, 62)


,sensor_id,box_id,title,unit,sensor_type,icon,last_measurement_createdAt,last_measurement_value,last_measurement_createdAt_year,last_measurement_createdAt_month,last_measurement_createdAt_day,last_measurement_createdAt_hour,last_measurement_createdAt_minute,last_measurement_createdAt_day_of_week,measurement_created_at,measurement_updated_at,device_name,exposure,model,description,image,last_measurement_at,grouptag,location_type,timestamp,weblink,location_geojson_type,location_geojson_geometry.coordinates,location_geojson_geometry.type,location_geojson_geometry.timestamp,current_location_longitude,current_location_latitude,measurement_created_at_year,measurement_created_at_month,measurement_created_at_day,measurement_created_at_hour,measurement_created_at_minute,measurement_created_at_day_of_week,measurement_updated_at_year,measurement_updated_at_month,measurement_updated_at_day,measurement_updated_at_hour,measurement_updated_at_minute,measurement_updated_at_day_of_week,last_measurement_at_year,last_measurement_at_month,last_measurement_at_day,last_measurement_at_hour,last_measurement_at_minute,last_measurement_at_day_of_week,timestamp_year,timestamp_month,timestamp_day,timestamp_hour,timestamp_minute,timestamp_day_of_week,location_geojson_geometry_timestamp_year,location_geojson_geometry_timestamp_month,location_geojson_geometry_timestamp_day,location_geojson_geometry_timestamp_hour,location_geojson_geometry_timestamp_minute,location_geojson_geometry_timestamp_day_of_week
0,578207d56fea661300861f3d,578207d56fea661300861f3b,UV intensity,μW/cm²,VEML6070,NaN,2026-11-18 16:27:47.560000+00:00,0.00,2026.0,11.0,18.0,16.0,27.0,2.0,2022-03-30 11:25:43.223000+00:00,2026-11-18 16:27:47.580000+00:00,receptive weather,outdoor,homeEthernet,"The measured values are processed by an Arduino Uno. Via a Modbus TCP interface, the measured value is transmitted to a Mini PLC. This PLC then transmits this and other data as a JSON array in the 60s clock.",578207d56fea661300861f3b_ovyk87.jpg,2026-11-18 16:27:47.560000+00:00,[],Point,2016-07-10 08:31:17+00:00,NaN,Feature,"[9.391277, 51.660264]",Point,2016-07-10 08:31:17+00:00,9.391277,51.660264,2022,3,30,11,25,2,2026,11,18,16,27,2,2026,11,18,16,27,2,2016,7,10,8,31,6,2016,7,10,8,31,6
1,578207d56fea661300861f3e,578207d56fea661300861f3b,Illuminance,lx,TSL45315,NaN,2026-11-18 16:27:47.560000+00:00,0.00,2026.0,11.0,18.0,16.0,27.0,2.0,2022-03-30 11:25:43.223000+00:00,2026-11-18 16:27:47.580000+00:00,receptive weather,outdoor,homeEthernet,"The measured values are processed by an Arduino Uno. Via a Modbus TCP interface, the measured value is transmitted to a Mini PLC. This PLC then transmits this and other data as a JSON array in the 60s clock.",578207d56fea661300861f3b_ovyk87.jpg,2026-11-18 16:27:47.560000+00:00,[],Point,2016-07-10 08:31:17+00:00,NaN,Feature,"[9.391277, 51.660264]",Point,2016-07-10 08:31:17+00:00,9.391277,51.660264,2022,3,30,11,25,2,2026,11,18,16,27,2,2026,11,18,16,27,2,2016,7,10,8,31,6,2016,7,10,8,31,6
2,578207d56fea661300861f3f,578207d56fea661300861f3b,Air pressure,hPa,BMP280,NaN,2026-11-18 16:27:47.560000+00:00,975.00,2026.0,11.0,18.0,16.0,27.0,2.0,2022-03-30 11:25:43.223000+00:00,2026-11-18 16:27:47.580000+00:00,receptive weather,outdoor,homeEthernet,"The measured values are processed by an Arduino Uno. Via a Modbus TCP interface, the measured value is transmitted to a Mini PLC. This PLC then transmits this and other data as a JSON array in the 60s clock.",578207d56fea661300861f3b_ovyk87.jpg,2026-11-18 16:27:47.560000+00:00,[],Point,2016-07-10 08:31:17+00:00,NaN,Feature,"[9.391277, 51.660264]",Point,2016-07-10 08:31:17+00:00,9.391277,51.660264,2022,3,30,11,25,2,2026,11,18,16,27,2,2026,11,18,16,27,2,2016,7,10,8,31,6,2016,7,10,8,31,6
3,578207d56fea661300861f40,578207d56fea661300861f3b,Relative humidity,%,HDC1008,NaN,2026-11-18 16:27:47.560000+00:00,95.98,2026.0,11.0,18.0,16.0,27.0,2.0,2022-03-30 11:25:43.223000+00:00,2026-11-18 16:27:47.580000+00:00,receptive weather,outdoor,homeEthernet,"Th

In [59]:
def save_iot_catalog(
    df: pd.DataFrame,
    output_dir: str = "data/processed",
    filename: str = "iot_sensor_catalog.parquet",
    overwrite: bool = True
) -> Path:
    """
    Save the combined IoT sensor catalog as a Parquet file.

    Supports idempotent execution through controlled overwriting.
    Validates the dataset before saving and verifies the output.
    """

    if df.empty:
        raise ValueError("Cannot save an empty IoT sensor catalog.")

    required_columns = ["sensor_id", "box_id"]

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise KeyError(f"Missing required columns: {missing_columns}")

    if df["sensor_id"].isna().any():
        raise ValueError("The catalog contains null sensor_id values.")

    if df["sensor_id"].duplicated().any():
        raise ValueError("The catalog contains duplicate sensor IDs.")

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    file_path = output_path / filename

    # Prevent accidental replacement when overwrite is disabled.
    if file_path.exists() and not overwrite:
        raise FileExistsError(
            f"Output file already exists: {file_path}. "
            "Set overwrite=True to replace it."
        )

    temp_path = file_path.with_name(
        f"{file_path.stem}.tmp{file_path.suffix}"
    )

    try:
        # Write the new dataset to a temporary file.
        df.to_parquet(
            temp_path,
            engine="pyarrow",
            compression="snappy",
            index=False
        )

        # Validate the temporary file.
        saved_df = pd.read_parquet(temp_path)

        if len(saved_df) != len(df):
            raise ValueError("Saved record count does not match source.")

        if list(saved_df.columns) != list(df.columns):
            raise ValueError("Saved columns do not match source.")

        if not saved_df["sensor_id"].is_unique:
            raise ValueError("Saved dataset contains duplicate sensor IDs.")

        # Replace the existing file only after validation succeeds.
        temp_path.replace(file_path)

    except Exception:
        if temp_path.exists():
            temp_path.unlink()
        raise

    print("IoT sensor catalog saved successfully.")
    print("Overwrite enabled:", overwrite)
    print("Output file:", file_path.resolve())
    print(f"Records: {len(df):,}")
    print(f"Columns: {len(df.columns):,}")
    print(f"File size: {file_path.stat().st_size / (1024 * 1024):.2f} MB")

    return file_path

In [60]:
iot_catalog_path = save_iot_catalog(
    df=iot_sensor_catalog_df,
    output_dir="data/processed",
    filename="iot_sensor_catalog.parquet",
    overwrite=True
)

IoT sensor catalog saved successfully.
Overwrite enabled: True
Output file: /home/collonel/data-engineering/iot_sensor_etl/data/processed/iot_sensor_catalog.parquet
Records: 13,584
Columns: 62
File size: 0.91 MB
